In [3]:
using CSV, DataFrames
using Dates
using TimeZones
using JuMP
using Gurobi
using Printf
using Statistics
using Dates
using TimeZones

# Parse strings like:
#  - "2025-01-01 19:00:00+00:00"
#  - "2025-01-01T19:00:00Z"
#  - "2025-01-01 19:00:00"
function parse_zdt_any(s::AbstractString; default_tz::TimeZone=tz"UTC")
    s2 = strip(s)

    # Try timezone-aware formats first (preferred)
    # Handles "+00:00" offsets and "Z" if the string is ISO-ish
    try
        # If it has an explicit offset or Z, ZonedDateTime can parse it.
        # Many CSVs use space instead of 'T', so normalize:
        s_iso = replace(s2, " " => "T")
        # If it ends with "+00:00" etc, this should work:
        return ZonedDateTime(s_iso)
    catch
        # fall through
    end

    # If no offset present, parse as naive DateTime using common patterns,
    # then attach default_tz.
    fmts = (
        dateformat"yyyy-mm-ddTHH:MM:SS",
        dateformat"yyyy-mm-ddTHH:MM:SS.s",
        dateformat"yyyy-mm-ddTHH:MM:SS.ss",
        dateformat"yyyy-mm-ddTHH:MM:SS.sss",
        dateformat"yyyy-mm-dd HH:MM:SS",
        dateformat"m/d/y H:MM",
        dateformat"m/d/y H:MM:SS",
        dateformat"m/d/y I:MM:SS p",
        dateformat"m/d/y I:MM p"
    )
    for f in fmts
        try
            dt = DateTime(s2, f)
            return ZonedDateTime(dt, default_tz)
        catch
        end
    end

    error("Invalid DateTime string: '$s2'")
end

# Column helpers
parse_utc_col(col) = astimezone.(parse_zdt_any.(String.(col); default_tz=tz"UTC"), tz"UTC")
parse_tz_col(col, tz::TimeZone) = parse_zdt_any.(String.(col); default_tz=tz)

# 0) Paths

BASE = pwd()

DA_CSV_PATH      = joinpath(BASE, "DA_LMPs_Data/PJM/df_DA_PJM_AECO_2025_DA_hourly.csv")
RT_CSV_PATH      = joinpath(BASE, "RT_LMPs_Data/PJM/df_RT_PJM_AECO_2025_RT_5min.csv")
REG_CSV_PATH     = joinpath(BASE, "reg_zone_prelim_bill_PJM_January_2025.csv")
SIGNALS_CSV_PATH = joinpath(BASE, "AGC Signal for Regulation/PJM/01_2025_avg_5min_pos_neg.csv")
EV_CSV_PATH      = joinpath(BASE, "EV_fleet.csv")

# 1) Time window

TZ = tz"America/New_York"
pt_start = ZonedDateTime(DateTime(2025, 1, 16, 14, 0, 0), TZ)
pt_end   = ZonedDateTime(DateTime(2025, 1, 17, 14, 0, 0), TZ)

utc_start = astimezone(pt_start, tz"UTC")
utc_end   = astimezone(pt_end,   tz"UTC")

# 2) Load datasets

df_DA_all  = CSV.read(DA_CSV_PATH, DataFrame)
df_RT_all  = CSV.read(RT_CSV_PATH, DataFrame)
df_REG_all = CSV.read(REG_CSV_PATH, DataFrame)
df_EV      = CSV.read(EV_CSV_PATH, DataFrame)

# Parse timestamps (DA/RT are UTC)
#df_DA_all.interval_start_utc = ZonedDateTime.(DateTime.(df_DA_all.interval_start_utc), tz"UTC")
#df_RT_all.interval_start_utc = ZonedDateTime.(DateTime.(df_RT_all.interval_start_utc), tz"UTC")
df_DA_all.interval_start_utc = parse_utc_col(df_DA_all.interval_start_utc)
df_RT_all.interval_start_utc = parse_utc_col(df_RT_all.interval_start_utc)

df_DA = df_DA_all[(df_DA_all.interval_start_utc .>= utc_start) .&
                  (df_DA_all.interval_start_utc .<  utc_end), :]
sort!(df_DA, :interval_start_utc)
df_DA = copy(df_DA)

df_RT = df_RT_all[(df_RT_all.interval_start_utc .>= utc_start) .&
                  (df_RT_all.interval_start_utc .<  utc_end), :]
sort!(df_RT, :interval_start_utc)
df_RT = copy(df_RT)

# REG is EPT in your file
#df_REG_all.datetime_beginning_ept = ZonedDateTime.(DateTime.(df_REG_all.datetime_beginning_ept), TZ)
df_REG_all.datetime_beginning_ept = parse_tz_col(df_REG_all.datetime_beginning_ept, TZ)

df_REG = df_REG_all[(df_REG_all.datetime_beginning_ept .>= pt_start) .&
                    (df_REG_all.datetime_beginning_ept .<  pt_end), :]
sort!(df_REG, :datetime_beginning_ept)
df_REG = copy(df_REG)

# =========================
# 3) EV parameter dictionaries
# =========================
EV_ids = Vector{String}(df_EV.EV)

E_bat_max = Dict(df_EV.EV .=> Float64.(df_EV[!, "E_bat_max (kWh)"]))
n_eff     = Dict(df_EV.EV .=> Float64.(df_EV[!, "n"]))
Pchrg     = Dict(df_EV.EV .=> Float64.(df_EV[!, "Pchrg (kW)"]))
SOEa      = Dict(df_EV.EV .=> Float64.(df_EV[!, "SOEa"]))
SOEd      = Dict(df_EV.EV .=> Float64.(df_EV[!, "SOEd"]))

Ta_hr     = Dict(df_EV.EV .=> Float64.(df_EV[!, "Ta_hr"]))
Td_hr     = Dict(df_EV.EV .=> Float64.(df_EV[!, "Td_hr"]))

# =========================
# 4) Indexing (match your Pyomo)
# =========================
T  = 24
K  = 12
NK = T * K

t0 = 14
k0 = t0 * K

IDX_t = collect(t0:(t0 + T - 1))         # 14..37
IDX_k = collect(k0:(k0 + NK - 1))        # 168..455
IDX_w = ["w1"]                           # single scenario

# Prices to dictionaries
@assert nrow(df_DA) >= T  "DA window has fewer than T=24 rows"
@assert nrow(df_RT) >= NK "RT window has fewer than NK=288 rows"
@assert nrow(df_REG) >= T "REG window has fewer than T=24 rows"

lmp_DA = Dict(IDX_t[i] => Float64(df_DA.lmp[i]) for i in 1:T)
lmp_RT = Dict(IDX_k[i] => Float64(df_RT.lmp[i]) for i in 1:NK)

# Hourly RT averages over 12×5-min windows
lt_RTE = Dict{Int,Float64}()
for (i, t) in enumerate(IDX_t)
    s = (i - 1) * K + 1
    e = i * K
    lt_RTE[t] = mean(Float64.(df_RT.lmp[s:e]))
end

# Regulation prices (hourly)
lt_RMCCP = Dict(IDX_t[i] => Float64(df_REG.rmccp[i]) for i in 1:T)  # capacity
lt_RMPCP = Dict(IDX_t[i] => Float64(df_REG.rmpcp[i]) for i in 1:T)  # mileage

# =========================
# 5) RT signals (5-min)
# =========================
RT_Signals = CSV.read(SIGNALS_CSV_PATH, DataFrame)
pos_col = "1/16/2025_pos"
neg_col = "1/16/2025_neg"

RT_Signals[!, pos_col] = coalesce.(RT_Signals[!, pos_col], 0.0)
RT_Signals[!, neg_col] = coalesce.(RT_Signals[!, neg_col], 0.0)

# Treat both columns as POSITIVE magnitudes (common convention in your Pyomo construction)
r_pos = Dict(IDX_k[i] => Float64(RT_Signals[i, pos_col]) for i in 1:NK)
#r_neg = Dict(IDX_k[i] => Float64(RT_Signals[i, neg_col]) for i in 1:NK)
r_neg = Dict(IDX_k[i] => abs(Float64(RT_Signals[i, neg_col])) for i in 1:NK)

hour_from_k(k) = t0 + div(k - k0, K)

# 6) Availability u(i,k) ∈ {0,1}

Ta_k = Dict(i => Int(round(Ta_hr[i] * K)) for i in EV_ids)
Td_k = Dict(i => Int(round(Td_hr[i] * K)) for i in EV_ids)

u = Dict{Tuple{String,Int},Int}()
for i in EV_ids, k in IDX_k
    u[(i,k)] = (k >= Ta_k[i] && k < Td_k[i]) ? 1 : 0
end

# 7) DART penalty M

dev = 0.3
max_gap = maximum(max(0.0, lt_RTE[t] - lmp_DA[t]) for t in IDX_t)
M_fixed = 1.01 / (1.0 - dev) * max_gap

# 8) Build JuMP model

m = Model(Gurobi.Optimizer)
set_optimizer_attribute(m, "MIPGap", 1e-4)
set_optimizer_attribute(m, "TimeLimit", 300.0)

Dt = 1 / 12
s_perf = Dict(t => 0.985 for t in IDX_t)
a_mil  = Dict(t => 1.0   for t in IDX_t)

# ---- Variables
@variable(m, E_DA[t in IDX_t] >= 0)                              # MWh
@variable(m, E_RT[k in IDX_k, w in IDX_w] >= 0)                  # MWh
@variable(m, E_i_RT[i in EV_ids, k in IDX_k, w in IDX_w] >= 0)   # kWh
@variable(m, 0 <= SOE[i in EV_ids, k in IDX_k, w in IDX_w] <= 1) # fraction
@variable(m, P_RT_max[i in EV_ids, k in IDX_k, w in IDX_w] >= 0) # kW

@variable(m, DE[t in IDX_t, w in IDX_w])                         # MWh (signed)
@variable(m, DE_U[t in IDX_t, w in IDX_w])
@variable(m, DE_I[t in IDX_t, w in IDX_w])

@variable(m, DE_U_Up[t in IDX_t, w in IDX_w] >= 0)
@variable(m, DE_U_Down[t in IDX_t, w in IDX_w] >= 0)

@variable(m, Pnlty_UUp[t in IDX_t, w in IDX_w] >= 0)
@variable(m, Pnlty_UDown[t in IDX_t, w in IDX_w] >= 0)

@variable(m, b[t in IDX_t, w in IDX_w], Bin)

@variable(m, R_RT_pos[t in IDX_t, w in IDX_w] >= 0)   # MW
@variable(m, R_RT_neg[t in IDX_t, w in IDX_w] >= 0)   # MW

@variable(m, E_RT_R_neg[k in IDX_k, w in IDX_w] >= 0) # MWh
@variable(m, E_RT_R_pos[k in IDX_k, w in IDX_w] >= 0) # MWh

BIGM = 1e6
SOEcc_cv = 0.85

# ---- Constraints

# DE[t,w] == E_DA[t] - sum_k E_RT[k,w] over hour
for t in IDX_t, w in IDX_w
    k0_t = t * K
    k1_t = (t + 1) * K - 1
    @constraint(m, DE[t, w] == E_DA[t] - sum(E_RT[k, w] for k in k0_t:k1_t))
end

@constraint(m, [t in IDX_t, w in IDX_w], DE[t,w] == DE_U[t,w] + DE_I[t,w])
@constraint(m, [t in IDX_t, w in IDX_w], DE_U[t,w] == DE_U_Up[t,w] - DE_U_Down[t,w])

# DE_I = sum (E_pos - E_neg) over hour
for t in IDX_t, w in IDX_w
    k0_t = t * K
    k1_t = (t + 1) * K - 1
    @constraint(m, DE_I[t, w] == sum(E_RT_R_pos[k, w] - E_RT_R_neg[k, w] for k in k0_t:k1_t))
end

# Deployment energies from ratios & awarded regulation MW
@constraint(m, [k in IDX_k, w in IDX_w],
    E_RT_R_pos[k,w] == r_pos[k] * R_RT_pos[hour_from_k(k), w] * Dt
)

@constraint(m, [k in IDX_k, w in IDX_w],
    E_RT_R_neg[k,w] == r_neg[k] * R_RT_neg[hour_from_k(k), w] * Dt
)

# R_pos bound
@constraint(m, [t in IDX_t, w in IDX_w],
    R_RT_pos[t,w] <= E_DA[t] - DE_U[t,w]
)

# R_neg bound (fleet capability minus scheduled net)
for t in IDX_t, w in IDX_w
    k0_t = t * K
    k1_t = (t + 1) * K - 1
    @constraint(m,
        R_RT_neg[t, w] <=
            sum(u[(i, k)] * P_RT_max[i, k, w] for i in EV_ids for k in k0_t:k1_t)
            - (E_DA[t] - DE_U[t, w])
    )
end

# Big-M logic for DE_U split
@constraint(m, [t in IDX_t, w in IDX_w], DE_U_Up[t,w]   <= BIGM * b[t,w])
@constraint(m, [t in IDX_t, w in IDX_w], DE_U_Down[t,w] <= BIGM * (1 - b[t,w]))
@constraint(m, [t in IDX_t, w in IDX_w], DE[t,w] >= -BIGM * (1 - b[t,w]))
@constraint(m, [t in IDX_t, w in IDX_w], DE[t,w] <=  BIGM * b[t,w])

# Aggregation: E_RT[k] == sum_i E_i_RT / 1000
@constraint(m, [k in IDX_k, w in IDX_w],
    E_RT[k,w] == sum(E_i_RT[i,k,w] for i in EV_ids) / 1000.0
)

# E_i_RT ≤ P_RT_max * Dt
@constraint(m, [i in EV_ids, k in IDX_k, w in IDX_w],
    E_i_RT[i,k,w] <= P_RT_max[i,k,w] * Dt
)

# P_RT_max ≤ u * Pchrg
@constraint(m, [i in EV_ids, k in IDX_k, w in IDX_w],
    P_RT_max[i,k,w] <= u[(i,k)] * Pchrg[i]
)

# taper as SOE approaches SOEcc_cv
@constraint(m, [i in EV_ids, k in IDX_k, w in IDX_w],
    P_RT_max[i,k,w] <= u[(i,k)] * Pchrg[i] * ((1 - SOE[i,k,w]) / (1 - SOEcc_cv))
)

# Initial SOE at k0
@constraint(m, [i in EV_ids, w in IDX_w], SOE[i, k0, w] == SOEa[i])

# SOE dynamics
@constraint(m, [i in EV_ids, k in IDX_k[1:end-1], w in IDX_w],
    SOE[i, k + 1, w] == SOE[i, k, w] + (n_eff[i] / E_bat_max[i]) * E_i_RT[i, k, w]
)

# Departure SOE target at k_dep = Td_k[i] - 1 (only if within horizon)
for i in EV_ids, w in IDX_w
    k_dep = Td_k[i] - 1
    if (k0 <= k_dep <= (k0 + NK - 1))
        @constraint(m, SOE[i, k_dep, w] == SOEd[i])
    end
end

# Penalty
@constraint(m, [t in IDX_t, w in IDX_w],
    Pnlty_UUp[t,w] >= M_fixed * (DE_U_Up[t,w]   - dev * E_DA[t])
)
@constraint(m, [t in IDX_t, w in IDX_w],
    Pnlty_UDown[t,w] >= M_fixed * (DE_U_Down[t,w] - dev * E_DA[t])
)

# 9) Objective

w = "w1"
@objective(m, Max, sum(
    -lmp_DA[t] * E_DA[t]
    + lt_RMCCP[t] * (R_RT_pos[t,w] + R_RT_neg[t,w]) * s_perf[t]
    + lt_RMPCP[t] * (R_RT_pos[t,w] + R_RT_neg[t,w]) * s_perf[t] * a_mil[t]
    + lt_RTE[t]   * DE[t,w]
    - Pnlty_UUp[t,w] - Pnlty_UDown[t,w]
    for t in IDX_t
))

# 10) Solve + output

optimize!(m)

term = termination_status(m)
primal = primal_status(m)
@printf("Termination: %s | Primal: %s\n", string(term), string(primal))

# Compare
term_s = string(term)
if term_s == "OPTIMAL" || term_s == "TIME_LIMIT"
    @printf("Total Profit: \$%.2f\n\n", objective_value(m))

    out = DataFrame(
        "t" => IDX_t,
        "DA_USD/MWh" => [lmp_DA[t] for t in IDX_t],
        "RTavg_UDS/MWh" => [lt_RTE[t] for t in IDX_t],
        "Reg_RMCCP_USD/MW" => [lt_RMCCP[t] for t in IDX_t],
        "E_DA_MWh" => [value(E_DA[t]) for t in IDX_t],
        "DE_MWh" => [value(DE[t, w]) for t in IDX_t],
        "R_RTpos_MW" => [value(R_RT_pos[t, w]) for t in IDX_t],
        "R_RTneg_MW" => [value(R_RT_neg[t, w]) for t in IDX_t],
     )

show(first(out, 24), allcols=true)
println()


    show(first(out, 24), allcols=true)
    println()
else
    println("Solve did not succeed. Status: ", term_s)
end


Set parameter Username
Set parameter LicenseID to value 2770245
Academic license - for non-commercial use only - expires 2027-01-26
Set parameter MIPGap to value 0.0001
Set parameter TimeLimit to value 300
Set parameter MIPGap to value 0.0001
Set parameter TimeLimit to value 300
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11.0 (26100.2))

CPU model: Intel(R) Core(TM) Ultra 7 165U, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 14 logical processors, using up to 14 threads

Non-default parameters:
TimeLimit  300

Optimize a model with 231752 rows, 173928 columns and 524968 nonzeros (Max)
Model fingerprint: 0x36362ae7
Model has 144 linear objective coefficients
Variable types: 173904 continuous, 24 integer (24 binary)
Coefficient statistics:
  Matrix range     [3e-04, 1e+06]
  Objective range  [1e+00, 1e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e-01, 1e+06]
Presolve removed 230547 rows and 172708 columns
Presolve time: 1.67s
Preso

In [3]:
using CSV, DataFrames
using Dates
using TimeZones
using JuMP
using Gurobi
using Printf
using Statistics

# =========================================================
# 1) Robust datetime parsing helpers
# =========================================================
function parse_zdt_any(s::AbstractString; default_tz::TimeZone=tz"UTC")
    s2 = strip(s)

    # Try timezone-aware ISO-ish first
    try
        s_iso = replace(s2, " " => "T")
        return ZonedDateTime(s_iso)
    catch
    end

    # Fall back to naive DateTime formats, then attach default_tz
    fmts = (
        dateformat"yyyy-mm-ddTHH:MM:SS",
        dateformat"yyyy-mm-ddTHH:MM:SS.s",
        dateformat"yyyy-mm-ddTHH:MM:SS.ss",
        dateformat"yyyy-mm-ddTHH:MM:SS.sss",
        dateformat"yyyy-mm-dd HH:MM:SS",
        dateformat"m/d/y H:MM",
        dateformat"m/d/y H:MM:SS",
        dateformat"m/d/y I:MM:SS p",
        dateformat"m/d/y I:MM p"
    )
    for f in fmts
        try
            dt = DateTime(s2, f)
            return ZonedDateTime(dt, default_tz)
        catch
        end
    end

    error("Invalid DateTime string: '$s2'")
end

parse_utc_col(col) = astimezone.(parse_zdt_any.(String.(col); default_tz=tz"UTC"), tz"UTC")
parse_tz_col(col, tz::TimeZone) = parse_zdt_any.(String.(col); default_tz=tz)

# =========================================================
# 2) Paths + load data ONCE
# =========================================================
BASE = pwd()

DA_CSV_PATH      = joinpath(BASE, "DA_LMPs_Data/PJM/df_DA_PJM_AECO_2025_DA_hourly.csv")
RT_CSV_PATH      = joinpath(BASE, "RT_LMPs_Data/PJM/df_RT_PJM_AECO_2025_RT_5min.csv")
REG_CSV_PATH     = joinpath(BASE, "reg_zone_prelim_bill_PJM_January_2025.csv")
SIGNALS_CSV_PATH = joinpath(BASE, "AGC Signal for Regulation/PJM/01_2025_avg_5min_pos_neg.csv")
EV_CSV_PATH      = joinpath(BASE, "EV_fleet.csv")

df_DA_all  = CSV.read(DA_CSV_PATH, DataFrame)
df_RT_all  = CSV.read(RT_CSV_PATH, DataFrame)
df_REG_all = CSV.read(REG_CSV_PATH, DataFrame)
df_EV      = CSV.read(EV_CSV_PATH, DataFrame)
RT_Signals = CSV.read(SIGNALS_CSV_PATH, DataFrame)

# parse timestamps once
df_DA_all.interval_start_utc = parse_utc_col(df_DA_all.interval_start_utc)
df_RT_all.interval_start_utc = parse_utc_col(df_RT_all.interval_start_utc)

TZ = tz"America/New_York"
df_REG_all.datetime_beginning_ept = parse_tz_col(df_REG_all.datetime_beginning_ept, TZ)

# =========================================================
# 3) EV parameter dictionaries (once)
# =========================================================
EV_ids = Vector{String}(df_EV.EV)

E_bat_max = Dict(df_EV.EV .=> Float64.(df_EV[!, "E_bat_max (kWh)"]))
n_eff     = Dict(df_EV.EV .=> Float64.(df_EV[!, "n"]))
Pchrg     = Dict(df_EV.EV .=> Float64.(df_EV[!, "Pchrg (kW)"]))
SOEa      = Dict(df_EV.EV .=> Float64.(df_EV[!, "SOEa"]))
SOEd      = Dict(df_EV.EV .=> Float64.(df_EV[!, "SOEd"]))
Ta_hr     = Dict(df_EV.EV .=> Float64.(df_EV[!, "Ta_hr"]))
Td_hr     = Dict(df_EV.EV .=> Float64.(df_EV[!, "Td_hr"]))

# =========================================================
# 4) Model runner for ONE day horizon (14:00 -> next day 14:00)
#    Fixes:
#      - rename argument to avoid shadowing Dates.day()
#      - use Dates.day(dte) explicitly
#      - return consistent outputs
# =========================================================
function run_horizon_day(
    dte::Date;
    t0::Int=14,            # your Pyomo-style index start hour
    T::Int=24,
    K::Int=12,
    dev::Float64=0.3,
    mipgap::Float64=1e-4,
    timelimit::Float64=300.0
)
    NK = T*K
    k0 = t0*K

    IDX_t = collect(t0:(t0 + T - 1))
    IDX_k = collect(k0:(k0 + NK - 1))
    IDX_w = ["w1"]

    # local window: dte 14:00 -> (dte+1) 14:00 in America/New_York
    pt_start = ZonedDateTime(DateTime(year(dte), month(dte), Dates.day(dte), 14, 0, 0), TZ)
    pt_end   = pt_start + Hour(24)

    utc_start = astimezone(pt_start, tz"UTC")
    utc_end   = astimezone(pt_end,   tz"UTC")

    # Slice DA/RT/REG to this horizon
    df_DA = df_DA_all[(df_DA_all.interval_start_utc .>= utc_start) .&
                      (df_DA_all.interval_start_utc .<  utc_end), :]
    sort!(df_DA, :interval_start_utc); df_DA = copy(df_DA)

    df_RT = df_RT_all[(df_RT_all.interval_start_utc .>= utc_start) .&
                      (df_RT_all.interval_start_utc .<  utc_end), :]
    sort!(df_RT, :interval_start_utc); df_RT = copy(df_RT)

    df_REG = df_REG_all[(df_REG_all.datetime_beginning_ept .>= pt_start) .&
                        (df_REG_all.datetime_beginning_ept .<  pt_end), :]
    sort!(df_REG, :datetime_beginning_ept); df_REG = copy(df_REG)

    @assert nrow(df_DA) >= T  "DA window has fewer than T=$T rows for date=$dte"
    @assert nrow(df_RT) >= NK "RT window has fewer than NK=$NK rows for date=$dte"
    @assert nrow(df_REG) >= T "REG window has fewer than T=$T rows for date=$dte"

    # Prices dictionaries
    lmp_DA = Dict(IDX_t[i] => Float64(df_DA.lmp[i]) for i in 1:T)
    lmp_RT = Dict(IDX_k[i] => Float64(df_RT.lmp[i]) for i in 1:NK)

    lt_RTE = Dict{Int,Float64}()
    for (i, t) in enumerate(IDX_t)
        s = (i - 1) * K + 1
        e = i * K
        lt_RTE[t] = mean(Float64.(df_RT.lmp[s:e]))
    end

    lt_RMCCP = Dict(IDX_t[i] => Float64(df_REG.rmccp[i]) for i in 1:T)
    lt_RMPCP = Dict(IDX_t[i] => Float64(df_REG.rmpcp[i]) for i in 1:T)

    # ---- Regulation signals: pick columns based on this date
    mdY = Dates.format(dte, "m/d/yyyy")
    pos_col = string(mdY, "_pos")
    neg_col = string(mdY, "_neg")

    if !(pos_col in names(RT_Signals)) || !(neg_col in names(RT_Signals))
        @printf("Skipping %s: signals columns not found: %s / %s\n",
                string(dte), pos_col, neg_col)
        return (status="SKIPPED_NO_SIGNALS", date=dte, profit=NaN, out=DataFrame())
    end

    RT_Signals_local = RT_Signals[:, [pos_col, neg_col]]
    RT_Signals_local[!, pos_col] = coalesce.(RT_Signals_local[!, pos_col], 0.0)
    RT_Signals_local[!, neg_col] = coalesce.(RT_Signals_local[!, neg_col], 0.0)

    @assert nrow(RT_Signals_local) >= NK "Signals file has fewer than NK=$NK rows for date=$dte"

    r_pos = Dict(IDX_k[i] => Float64(RT_Signals_local[i, pos_col]) for i in 1:NK)
    r_neg = Dict(IDX_k[i] => abs(Float64(RT_Signals_local[i, neg_col])) for i in 1:NK)

    hour_from_k(k) = t0 + div(k - k0, K)

    # Availability u(i,k)
    Ta_k = Dict(i => Int(round(Ta_hr[i] * K)) for i in EV_ids)
    Td_k = Dict(i => Int(round(Td_hr[i] * K)) for i in EV_ids)

    u = Dict{Tuple{String,Int},Int}()
    for i in EV_ids, k in IDX_k
        u[(i,k)] = (k >= Ta_k[i] && k < Td_k[i]) ? 1 : 0
    end

    # DART penalty M
    max_gap = maximum(max(0.0, lt_RTE[t] - lmp_DA[t]) for t in IDX_t)
    M_fixed = 1.01 / (1.0 - dev) * max_gap

    # =========================================================
    # Build JuMP model
    # =========================================================
    m = Model(Gurobi.Optimizer)
    set_optimizer_attribute(m, "MIPGap", mipgap)
    set_optimizer_attribute(m, "TimeLimit", timelimit)

    Dt = 1/12
    s_perf = Dict(t => 0.985 for t in IDX_t)
    a_mil  = Dict(t => 1.0   for t in IDX_t)

    @variable(m, E_DA[t in IDX_t] >= 0)                              # MWh
    @variable(m, E_RT[k in IDX_k, w in IDX_w] >= 0)                  # MWh
    @variable(m, E_i_RT[i in EV_ids, k in IDX_k, w in IDX_w] >= 0)   # kWh
    @variable(m, 0 <= SOE[i in EV_ids, k in IDX_k, w in IDX_w] <= 1)
    @variable(m, P_RT_max[i in EV_ids, k in IDX_k, w in IDX_w] >= 0) # kW

    @variable(m, DE[t in IDX_t, w in IDX_w])                         # MWh (signed)
    @variable(m, DE_U[t in IDX_t, w in IDX_w])
    @variable(m, DE_I[t in IDX_t, w in IDX_w])

    @variable(m, DE_U_Up[t in IDX_t, w in IDX_w] >= 0)
    @variable(m, DE_U_Down[t in IDX_t, w in IDX_w] >= 0)

    @variable(m, Pnlty_UUp[t in IDX_t, w in IDX_w] >= 0)
    @variable(m, Pnlty_UDown[t in IDX_t, w in IDX_w] >= 0)

    @variable(m, b[t in IDX_t, w in IDX_w], Bin)

    @variable(m, R_RT_pos[t in IDX_t, w in IDX_w] >= 0)   # MW
    @variable(m, R_RT_neg[t in IDX_t, w in IDX_w] >= 0)   # MW

    @variable(m, E_RT_R_neg[k in IDX_k, w in IDX_w] >= 0) # MWh
    @variable(m, E_RT_R_pos[k in IDX_k, w in IDX_w] >= 0) # MWh

    BIGM = 1e6
    SOEcc_cv = 0.85

    # ---- Constraints
    for t in IDX_t, w in IDX_w
        k0_t = t * K
        k1_t = (t + 1) * K - 1
        @constraint(m, DE[t, w] == E_DA[t] - sum(E_RT[k, w] for k in k0_t:k1_t))
    end

    @constraint(m, [t in IDX_t, w in IDX_w], DE[t,w] == DE_U[t,w] + DE_I[t,w])
    @constraint(m, [t in IDX_t, w in IDX_w], DE_U[t,w] == DE_U_Up[t,w] - DE_U_Down[t,w])

    for t in IDX_t, w in IDX_w
        k0_t = t * K
        k1_t = (t + 1) * K - 1
        @constraint(m, DE_I[t, w] == sum(E_RT_R_pos[k, w] - E_RT_R_neg[k, w] for k in k0_t:k1_t))
    end

    @constraint(m, [k in IDX_k, w in IDX_w],
        E_RT_R_pos[k,w] == r_pos[k] * R_RT_pos[hour_from_k(k), w] * Dt
    )
    @constraint(m, [k in IDX_k, w in IDX_w],
        E_RT_R_neg[k,w] == r_neg[k] * R_RT_neg[hour_from_k(k), w] * Dt
    )

    @constraint(m, [t in IDX_t, w in IDX_w],
        R_RT_pos[t,w] <= E_DA[t] - DE_U[t,w]
    )

    for t in IDX_t, w in IDX_w
        k0_t = t * K
        k1_t = (t + 1) * K - 1
        @constraint(m,
            R_RT_neg[t, w] <=
                sum(u[(i, k)] * P_RT_max[i, k, w] for i in EV_ids for k in k0_t:k1_t)
                - (E_DA[t] - DE_U[t, w])
        )
    end

    @constraint(m, [t in IDX_t, w in IDX_w], DE_U_Up[t,w]   <= BIGM * b[t,w])
    @constraint(m, [t in IDX_t, w in IDX_w], DE_U_Down[t,w] <= BIGM * (1 - b[t,w]))
    @constraint(m, [t in IDX_t, w in IDX_w], DE[t,w] >= -BIGM * (1 - b[t,w]))
    @constraint(m, [t in IDX_t, w in IDX_w], DE[t,w] <=  BIGM * b[t,w])

    @constraint(m, [k in IDX_k, w in IDX_w],
        E_RT[k,w] == sum(E_i_RT[i,k,w] for i in EV_ids) / 1000.0
    )

    @constraint(m, [i in EV_ids, k in IDX_k, w in IDX_w],
        E_i_RT[i,k,w] <= P_RT_max[i,k,w] * Dt
    )

    @constraint(m, [i in EV_ids, k in IDX_k, w in IDX_w],
        P_RT_max[i,k,w] <= u[(i,k)] * Pchrg[i]
    )

    @constraint(m, [i in EV_ids, k in IDX_k, w in IDX_w],
        P_RT_max[i,k,w] <= u[(i,k)] * Pchrg[i] * ((1 - SOE[i,k,w]) / (1 - SOEcc_cv))
    )

    @constraint(m, [i in EV_ids, w in IDX_w], SOE[i, k0, w] == SOEa[i])

    @constraint(m, [i in EV_ids, k in IDX_k[1:end-1], w in IDX_w],
        SOE[i, k + 1, w] == SOE[i, k, w] + (n_eff[i] / E_bat_max[i]) * E_i_RT[i, k, w]
    )

    for i in EV_ids, w in IDX_w
        k_dep = Td_k[i] - 1
        if (k0 <= k_dep <= (k0 + NK - 1))
            @constraint(m, SOE[i, k_dep, w] == SOEd[i])
        end
    end

    @constraint(m, [t in IDX_t, w in IDX_w],
        Pnlty_UUp[t,w] >= M_fixed * (DE_U_Up[t,w]   - dev * E_DA[t])
    )
    @constraint(m, [t in IDX_t, w in IDX_w],
        Pnlty_UDown[t,w] >= M_fixed * (DE_U_Down[t,w] - dev * E_DA[t])
    )

    w = "w1"
    @objective(m, Max, sum(
        -lmp_DA[t] * E_DA[t]
        + lt_RMCCP[t] * (R_RT_pos[t,w] + R_RT_neg[t,w]) * s_perf[t]
        + lt_RMPCP[t] * (R_RT_pos[t,w] + R_RT_neg[t,w]) * s_perf[t] * a_mil[t]
        + lt_RTE[t]   * DE[t,w]
        - Pnlty_UUp[t,w] - Pnlty_UDown[t,w]
        for t in IDX_t
    ))

    optimize!(m)

    term = termination_status(m)
    term_s = string(term)

    if term_s == "OPTIMAL" || term_s == "TIME_LIMIT"
        profit = objective_value(m)
        out = DataFrame(
            "date" => fill(string(dte), length(IDX_t)),
            "t" => IDX_t,
            "DA_USD/MWh" => [lmp_DA[t] for t in IDX_t],
            "RTavg_USD/MWh" => [lt_RTE[t] for t in IDX_t],
            "Reg_RMCCP_USD/MW" => [lt_RMCCP[t] for t in IDX_t],
            "Reg_RMPCP_USD/MW" => [lt_RMPCP[t] for t in IDX_t],
            "E_DA_MWh" => [value(E_DA[t]) for t in IDX_t],
            "DE_MWh" => [value(DE[t, w]) for t in IDX_t],
            "R_RTpos_MW" => [value(R_RT_pos[t, w]) for t in IDX_t],
            "R_RTneg_MW" => [value(R_RT_neg[t, w]) for t in IDX_t],
        )
        return (status=term_s, date=dte, profit=profit, out=out)
    else
        return (status=term_s, date=dte, profit=NaN, out=DataFrame())
    end
end

# =========================================================
# 5) Run ALL DAYS of a week
#    Note: in notebooks, if you re-run this cell after editing the function,
#    use Base.invokelatest to avoid world-age warnings.
# =========================================================
week_start = Date(2025, 1, 15)   # <-- change this
results = Dict{Date,DataFrame}()
summary = DataFrame(date=String[], status=String[], profit=Float64[])

for dd in 0:6
    dte = week_start + Day(dd)

    # safer in notebooks:
    r = Base.invokelatest(run_horizon_day, dte; t0=14, T=24, K=12)

    push!(summary, (string(dte), r.status, r.profit))

    if nrow(r.out) > 0
        results[dte] = r.out
        # CSV.write("out_$(string(dte)).csv", r.out)   # optional
    end

    @printf("%s | status=%s | profit=%s\n", string(dte), r.status, string(r.profit))
end

println("\n=== Weekly summary ===")
show(summary, allcols=true); println()

# optional: one combined file (only days that solved)
if !isempty(results)
    out_all = vcat(values(results)...)
    # CSV.write("out_week.csv", out_all)
end


Set parameter Username
Set parameter LicenseID to value 2770245
Academic license - for non-commercial use only - expires 2027-01-26
Set parameter MIPGap to value 0.0001
Set parameter TimeLimit to value 300
Set parameter MIPGap to value 0.0001
Set parameter TimeLimit to value 300
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11.0 (26100.2))

CPU model: Intel(R) Core(TM) Ultra 7 165U, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 14 logical processors, using up to 14 threads

Non-default parameters:
TimeLimit  300

Optimize a model with 231752 rows, 173928 columns and 524975 nonzeros (Max)
Model fingerprint: 0x5327b011
Model has 144 linear objective coefficients
Variable types: 173904 continuous, 24 integer (24 binary)
Coefficient statistics:
  Matrix range     [3e-04, 1e+06]
  Objective range  [1e+00, 2e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e-01, 1e+06]
Presolve removed 230547 rows and 172708 columns
Presolve time: 0.95s
Preso

Row,date,t,DA_USD/MWh,RTavg_USD/MWh,Reg_RMCCP_USD/MW,Reg_RMPCP_USD/MW,E_DA_MWh,DE_MWh,R_RTpos_MW,R_RTneg_MW
,String,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,2025-01-20,14,186.468,28.57,64.09,3.36,0.0,0.0,0.0,0.0
2,2025-01-20,15,193.511,36.1517,53.03,2.48,0.0,0.0,0.0,0.0
3,2025-01-20,16,204.855,81.3708,76.7,2.23,0.0,0.0,0.0,0.0
4,2025-01-20,17,276.41,163.535,178.35,1.68,0.0,0.0,0.0,0.0
5,2025-01-20,18,284.707,108.585,63.14,1.38,0.0,0.0,0.0,0.0
6,2025-01-20,19,271.287,109.069,66.69,1.61,0.0,0.0,0.0,0.0
7,2025-01-20,20,269.738,74.88,34.39,3.68,0.0,0.0,0.0,0.0
8,2025-01-20,21,261.012,75.0225,101.54,0.0,0.0,-0.469839,0.0,1.23614
9,2025-01-20,22,245.761,100.185,48.99,5.41,0.0,0.0,0.0,0.0
